<div align='center'>

# ⚡ KRONOS v6 — *Survive First, Dominate Second*

</div>

---

## 🔬 Root cause analysis:

| Problem | Fix |
|---------|-----|
| Lost home planets while attacking | **Defensive Floor**: keep max(10%, prod×2, incoming×1.15+5) |
| Suffocation = cosmetic | **Lethal Suffocation**: send `production×2+1` every 12 turns |
| Snipe timing imprecise | **1-Tick Snipe**: arrive exactly T+1 after enemy lands |
| Attacked when weaker | **Survival Gate**: only attack enemy if our_ships ≥ 1.5× nearest |
| Slow opening | **Initiative Bonus**: step<20 → garrison=4, max_atk=6 |

---


## ⚙️ Cell 1 — Install


In [ ]:
%%capture
!pip install --upgrade 'kaggle-environments>=1.28.0'


## 🔌 Cell 2 — Environment


In [ ]:
from kaggle_environments import make
import math
env=make('orbit_wars',debug=True)
env.reset()
obs=dict(env.state[0].observation)
print(f'✅ {env.name} v{env.version} | av={obs["angular_velocity"]:.4f} | planets={len(obs["planets"])}')


## ⚡ Cell 3 — KRONOS v6 Agent


In [ ]:
"""
KRONOS v6 — "Survive First, Dominate Second"

ROOT CAUSE OF v5 LOSS (10% win rate):
  • Garrison too low → lost home planets while attacking
  • Suffocation = cosmetic (3 ships useless vs prod-5 planet)
  • No survival check → attacked when weaker than enemy nearby
  • Snipe timing imprecise → arrived same time as enemy or too late

v6 FIXES:
  1. Defensive Floor    — keep max(10%, incoming+buffer) ships always
  2. Lethal Suffocation — send production×2+1 per cycle
  3. 1-Tick Snipe       — arrive exactly T+1 after enemy lands
  4. Survival Gate      — only attack if our_ships > 2× nearest_enemy
  5. Initiative Bonus   — if step<15, hyper-aggressive (opening theory)
"""
import math

SX,SY,SR,INNER,MS = 50.0,50.0,5.0,38.0,500

class _P:
    __slots__=['id','owner','x','y','radius','ships','production']
    def __init__(self,*a):
        for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)
class _F:
    __slots__=['id','owner','x','y','angle','ships']
    def __init__(self,*a):
        for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)

def spd(n):   return min(6.0,1.0+(max(1,n)-1)*5.0/99.0)
def d2(ax,ay,bx,by): return math.sqrt((ax-bx)**2+(ay-by)**2)
def inn(p):   return d2(p.x,p.y,SX,SY)<INNER
def pred(p,av,t):
    if not inn(p): return p.x,p.y
    r=d2(p.x,p.y,SX,SY); a=math.atan2(p.y-SY,p.x-SX)+av*t
    return SX+r*math.cos(a),SY+r*math.sin(a)
def icp(sx,sy,tp,av,n,it=20):
    tx,ty=tp.x,tp.y
    for _ in range(it):
        dd=d2(sx,sy,tx,ty); t=dd/spd(n) if spd(n)>0 else 1e9
        nx,ny=pred(tp,av,t)
        if d2(tx,ty,nx,ny)<0.015: break
        tx,ty=(tx+nx)/2,(ty+ny)/2
    dd=d2(sx,sy,tx,ty); t=dd/spd(n) if spd(n)>0 else 1e9
    return math.atan2(ty-sy,tx-sx),dd,t
def sun_ok(ox,oy,a,md):
    dx,dy=math.cos(a),math.sin(a); fx,fy=SX-ox,SY-oy; tp=fx*dx+fy*dy
    if not(0<tp<md): return True
    return abs(fx*dy-fy*dx)>=SR+1.5
def safe(ox,oy,a,d,sw=42,st=36):
    if sun_ok(ox,oy,a,d): return a,True
    for i in range(1,st+1):
        da=math.radians(sw)*i/st
        for s in(+1,-1):
            alt=a+s*da
            if sun_ok(ox,oy,alt,d): return alt,True
    return a,False

# ── Binary-search minimum capture ships ──────────────────────────────────────
def capture_n(tgt,av,sx,sy,buf=1.07):
    lo,hi=1,max(tgt.ships*2+20,30)
    for _ in range(16):
        mid=(lo+hi)//2
        _,_,eta=icp(sx,sy,tgt,av,mid)
        if mid>int((tgt.ships+tgt.production*eta)*buf):
            hi=mid
        else:
            lo=mid+1
    return hi

# ── FIX 1: DEFENSIVE FLOOR ───────────────────────────────────────────────────
def min_garrison(planet, stp, incoming_ships=0):
    """
    Keep at least:
      - incoming × 1.15 + 5 (if threatened)
      - max(10% of current ships, production×2, 4) (baseline)
    This prevents losing home planets while attacking.
    """
    if incoming_ships > 0:
        return int(incoming_ships * 1.15) + 5
    floor_10pct = max(4, int(planet.ships * 0.10))
    if stp > 380:
        return max(3, floor_10pct // 2)   # endgame: minimal
    if stp < 20:
        return 4                           # opening: very aggressive
    return max(floor_10pct, planet.production * 2, 4)

# ── FIX 4: SURVIVAL GATE ─────────────────────────────────────────────────────
def survival_ok(mine, planets, player, stp):
    """
    Allow full aggression only if:
      our_total_ships >= 1.5 × nearest_enemy_total
    In opening (stp<15): always OK.
    In endgame (stp>370): always OK.
    """
    if stp < 15 or stp > 370:
        return True

    my_total = sum(p.ships for p in mine)
    my_prod  = sum(p.production for p in mine)

    cx = sum(p.x for p in mine)/max(1,len(mine))
    cy = sum(p.y for p in mine)/max(1,len(mine))

    # Find nearest enemy cluster
    enemy_ids = set(p.owner for p in planets if p.owner>=0 and p.owner!=player)
    if not enemy_ids: return True

    nearest_enemy_power = 0
    for eid in enemy_ids:
        e_planets = [p for p in planets if p.owner==eid]
        if not e_planets: continue
        e_center_dist = min(d2(p.x,p.y,cx,cy) for p in e_planets)
        if e_center_dist < 50:   # only nearby enemies matter
            ep = sum(p.ships for p in e_planets)
            nearest_enemy_power = max(nearest_enemy_power, ep)

    return my_total >= nearest_enemy_power * 1.5 or my_prod >= 8

# ── FIX 2: LETHAL SUFFOCATION ────────────────────────────────────────────────
def lethal_suffocation(enemy, mine, av, used, stp):
    """
    Send production×2+1 ships every ~12 turns to highest-production enemy.
    Goal: their garrison never builds up enough to launch.
    """
    if stp < 60: return []
    if stp % 12 not in (0, 1): return []

    out = []
    targets = sorted([p for p in enemy if p.production >= 2],
                     key=lambda p: -p.production)[:2]

    for tgt in targets:
        n = tgt.production * 2 + 1   # lethal: resets their counter fully

        src = min(
            [p for p in mine
             if p.ships - used.get(p.id,0) - min_garrison(p,stp) >= n],
            key=lambda p: d2(p.x,p.y,tgt.x,tgt.y),
            default=None
        )
        if src is None: continue

        a,dd,_ = icp(src.x,src.y,tgt,av,n)
        sa,ok  = safe(src.x,src.y,a,dd)
        if ok:
            out.append((src.id, sa, n))

    return out

# ── FIX 3: 1-TICK SNIPE ──────────────────────────────────────────────────────
def one_tick_snipe(neutral, e_fleets, mine, av, used, done, enroute, stp):
    """
    Detect enemy fleet heading to neutral.
    Send our fleet to arrive exactly T+1 turns after enemy.
    Enemy spends ships capturing it → we take it at near-zero defense.
    """
    snipes = []

    for tgt in neutral:
        if tgt.id in done or tgt.id in enroute: continue

        incoming = []
        for f in e_fleets:
            a,dd,eta = icp(f.x,f.y,tgt,av,f.ships)
            if dd < tgt.radius+4 and eta < 60:
                incoming.append((eta, f.ships))

        if not incoming: continue
        incoming.sort()
        e_eta, e_ships = incoming[0]

        # After battle
        after_def = tgt.ships + tgt.production * e_eta
        if e_ships <= after_def: continue   # enemy doesn't capture

        enemy_leftover = max(0, e_ships - after_def)

        # We arrive T+1: face only enemy_leftover + 1 turn production
        our_arrival = e_eta + 1.0
        n_need = int(enemy_leftover * 1.1) + tgt.production + 2
        n_need = max(n_need, 3)

        for src in sorted(mine, key=lambda p: d2(p.x,p.y,tgt.x,tgt.y)):
            spare = src.ships - used.get(src.id,0) - min_garrison(src,stp)
            if spare < n_need: continue

            # Use Speed-as-Timing: compute n to arrive at our_arrival
            dist = d2(src.x,src.y,tgt.x,tgt.y)
            req_spd = max(1.0, min(6.0, dist / max(1,our_arrival)))
            n_timed = max(n_need, int(1 + (req_spd-1.0)*99/5))

            if n_timed > spare: n_timed = spare
            if n_timed < n_need: continue

            a,dd,our_eta = icp(src.x,src.y,tgt,av,n_timed)
            if our_eta < e_eta + 0.5: continue   # must arrive after enemy
            if our_eta > e_eta + 4:   continue   # not too late

            sa,ok = safe(src.x,src.y,a,dd)
            if ok:
                score = tgt.production*8 + e_ships*0.4
                snipes.append((score, src.id, sa, n_timed, tgt.id))
                break

    snipes.sort(key=lambda x:-x[0])
    return snipes

# ── Score function ────────────────────────────────────────────────────────────
def score(tgt, n, eta, rem):
    tw = max(0, rem-eta)
    if tw <= 0: return -1e9
    prod = tgt.production
    s = (prod**2)*12*tw + prod*tw
    if tgt.owner >= 0: s *= 1.7        # enemy: gain + deny
    if tgt.ships <= prod*2+2: s *= 2.0 # near-empty bonus
    s -= n*0.35
    return s

# ── Main agent ────────────────────────────────────────────────────────────────
def orbital_strategist(obs):
    if isinstance(obs,dict):
        pl=obs.get('player',0); rp=obs.get('planets',[])
        rf=obs.get('fleets',[]); av=obs.get('angular_velocity',0.0366)
        stp=obs.get('step',0)
    else:
        pl=obs.player; rp=obs.planets; rf=obs.fleets
        av=obs.angular_velocity; stp=getattr(obs,'step',0)

    try:
        from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as NP,Fleet as NF
        planets=[NP(*p) for p in rp]; fleets=[NF(*f) for f in rf]
    except Exception:
        planets=[_P(*p) for p in rp]; fleets=[_F(*f) for f in rf]

    mine    = [p for p in planets if p.owner==pl]
    neutral = [p for p in planets if p.owner<0]
    enemy   = [p for p in planets if p.owner>=0 and p.owner!=pl]
    others  = enemy + neutral
    if not mine or not others: return []

    rem=MS-stp; moves=[]; used={}; done=set()
    def avail(p):  return p.ships - used.get(p.id,0)
    def rsv(pid,n): used[pid]=used.get(pid,0)+n
    def spare(p):  return avail(p) - min_garrison(p,stp)

    # ── Incoming threats ──────────────────────────────────────────────────
    incoming = {}
    for f in fleets:
        if f.owner==pl: continue
        for p in mine:
            _,dd,eta = icp(f.x,f.y,p,av,f.ships)
            if dd < p.radius+spd(f.ships)*1.5+1 and eta < 20:
                incoming[p.id] = incoming.get(p.id,0) + f.ships

    # ── DEFENSE (always runs, no survival gate) ───────────────────────────
    for p in mine:
        thr = incoming.get(p.id,0)
        if thr == 0: continue
        need = min_garrison(p,stp,thr)
        deficit = need - avail(p)
        if deficit <= 0: continue
        donors = sorted(
            [s for s in mine if s.id!=p.id and spare(s)>4],
            key=lambda s: d2(s.x,s.y,p.x,p.y)
        )
        for src in donors[:3]:
            snd = min(spare(src), deficit)
            if snd <= 0: continue
            a,dd,_ = icp(src.x,src.y,p,av,snd)
            sa,ok  = safe(src.x,src.y,a,dd)
            if ok:
                moves.append([src.id,sa,snd])
                rsv(src.id,snd); deficit-=snd
            if deficit <= 0: break

    # ── En-route ──────────────────────────────────────────────────────────
    enroute = set()
    for f in fleets:
        if f.owner!=pl: continue
        for t in others:
            _,dd,eta = icp(f.x,f.y,t,av,f.ships)
            if dd < t.radius+4 and eta < 80:
                enroute.add(t.id)

    e_fleets = [f for f in fleets if f.owner!=pl and f.owner>=0]

    # ── FIX 3: 1-TICK SNIPE ──────────────────────────────────────────────
    for sc,sid,sa,n,tid in one_tick_snipe(neutral,e_fleets,mine,av,used,done,enroute,stp)[:2]:
        src = next((p for p in mine if p.id==sid),None)
        if src and spare(src) >= n:
            moves.append([sid,sa,n]); rsv(sid,n); done.add(tid)

    # ── RETROGRADE ────────────────────────────────────────────────────────
    for p in planets:
        if p.owner<0 or p.owner==pl: continue
        if p.id in done or p.id in enroute: continue
        dep = sum(f.ships for f in fleets
                  if f.owner==p.owner and d2(f.x,f.y,p.x,p.y)<14)
        if dep < 10: continue
        if dep/max(1,p.ships+dep) < 0.30: continue
        bsrc=None; bn=0; bsa=0.0
        for src in mine:
            sp = spare(src)
            if sp < 4: continue
            n = capture_n(p,av,src.x,src.y)
            if sp < n: continue
            a,dd,_ = icp(src.x,src.y,p,av,n)
            sa,ok  = safe(src.x,src.y,a,dd)
            if not ok: continue
            if bsrc is None or sp > bn: bsrc,bn,bsa=src,n,sa
        if bsrc:
            moves.append([bsrc.id,bsa,bn]); rsv(bsrc.id,bn); done.add(p.id)

    # ── FIX 2: LETHAL SUFFOCATION ─────────────────────────────────────────
    for sid,sa,n in lethal_suffocation(enemy,mine,av,used,stp):
        src = next((p for p in mine if p.id==sid),None)
        if src and spare(src) >= n:
            moves.append([sid,sa,n]); rsv(sid,n)

    # ── FIX 4: SURVIVAL GATE → main attack ────────────────────────────────
    can_attack = survival_ok(mine,planets,pl,stp)

    # ── SCORE ALL TARGETS ─────────────────────────────────────────────────
    cands=[]
    for src in mine:
        sp = spare(src)
        if sp < 3: continue
        for tgt in others:
            if tgt.id in done or tgt.id in enroute: continue
            n = capture_n(tgt,av,src.x,src.y)
            if n > sp: continue
            a,dd,eta = icp(src.x,src.y,tgt,av,n)
            sa,ok    = safe(src.x,src.y,a,dd)
            if not ok: continue
            sc = score(tgt,n,eta,rem)
            if sc > -1e8:
                cands.append((sc,id(src),src,tgt,n,sa))

    cands.sort(key=lambda x:-x[0])

    max_atk = 6 if (stp<80 or stp>380) else 4

    atks=0
    for sc,_,src,tgt,n,sa in cands:
        if atks >= max_atk: break
        if tgt.id in done or tgt.id in enroute: continue
        if spare(src) < n: continue

        # Survival gate: neutrals always OK, enemies only if strong enough
        if tgt.owner >= 0 and not can_attack: continue

        moves.append([src.id,sa,n]); rsv(src.id,n); done.add(tgt.id); atks+=1

    # ── SWEEP: zero idle ships ────────────────────────────────────────────
    for src in sorted(mine, key=lambda p:-spare(p)):
        sp = spare(src)
        if sp < 4: continue
        best=None; bsc=-1e9
        for tgt in others:
            if tgt.id in done: continue
            n = capture_n(tgt,av,src.x,src.y)
            if n > sp: continue
            a,dd,eta = icp(src.x,src.y,tgt,av,n)
            sa,ok    = safe(src.x,src.y,a,dd)
            if not ok: continue
            if tgt.owner>=0 and not can_attack: continue
            sc2 = tgt.production*10/(dd+1) + (1.6 if tgt.owner>=0 else 1.0)
            if sc2>bsc: bsc=sc2; best=(src.id,sa,n,tgt.id)
        if best:
            moves.append([best[0],best[1],best[2]])
            rsv(best[0],best[2]); done.add(best[3])

    return moves

agent = orbital_strategist


## 🔬 Cell 4 — v1 Baseline


In [ ]:
def v1_agent(obs):
    import math
    class _P:
        __slots__=['id','owner','x','y','radius','ships','production']
        def __init__(self,*a):
            for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)
    class _F:
        __slots__=['id','owner','x','y','angle','ships']
        def __init__(self,*a):
            for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)
    try:
        from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as _P,Fleet as _F
    except: pass
    def fs(n): return min(6.0,1.0+(max(1,n)-1)*5.0/99.0)
    def dd(ax,ay,bx,by): return math.sqrt((ax-bx)**2+(ay-by)**2)
    def isin(p): return dd(p.x,p.y,50,50)<38
    def pp(p,av,t):
        if not isin(p): return p.x,p.y
        r=dd(p.x,p.y,50,50); a=math.atan2(p.y-50,p.x-50)+av*t
        return 50+r*math.cos(a),50+r*math.sin(a)
    def icp2(sx,sy,tp,av,n):
        tx,ty=tp.x,tp.y
        for _ in range(15):
            d=dd(sx,sy,tx,ty); t=d/fs(n) if fs(n)>0 else 1e9
            nx,ny=pp(tp,av,t)
            if dd(tx,ty,nx,ny)<0.05: break
            tx,ty=nx,ny
        d=dd(sx,sy,tx,ty); return math.atan2(ty-sy,tx-sx),d,d/fs(n)
    def sh(ox,oy,a,md2):
        dx,dy=math.cos(a),math.sin(a); fx,fy=50-ox,50-oy; t=fx*dx+fy*dy
        return 0<t<md2 and abs(fx*dy-fy*dx)<6.5
    def sa2(ox,oy,a,d2):
        if not sh(ox,oy,a,d2): return a,True
        for i in range(1,13):
            dl=math.radians(25)*i/12
            for s in(1,-1):
                if not sh(ox,oy,a+s*dl,d2): return a+s*dl,True
        return a,False
    if isinstance(obs,dict):
        pl=obs.get('player',0);rp=obs.get('planets',[])
        rf=obs.get('fleets',[]); av=obs.get('angular_velocity',0.0366)
        stp=obs.get('step',250)
    else:
        pl=obs.player;rp=obs.planets;rf=obs.fleets
        av=obs.angular_velocity;stp=getattr(obs,'step',250)
    P=[_P(*p) for p in rp]; F=[_F(*f) for f in rf]
    mine=[p for p in P if p.owner==pl]; tgts=[p for p in P if p.owner!=pl]
    if not mine or not tgts: return []
    rem=500-stp; moves=[]; cmtd=set(); used={}
    def av2(p): return p.ships-used.get(p.id,0)
    for t in sorted([t for t in tgts if t.id not in cmtd
                     and t.ships<=t.production*4+3],key=lambda t:t.ships):
        bst=None; bs=-1e9
        for src in mine:
            if av2(src)<15: continue
            _,ddv,ta=icp2(src.x,src.y,t,av,t.ships+5)
            sc=t.production/(ddv+1)
            if sc>bs: bs=sc;bst=src;bta=ta
        if bst is None: continue
        n=max(int((t.ships+t.production*bta)*1.3)+1,int(t.ships*1.3)+5)
        if av2(bst)<n: continue
        ang,ddv,_=icp2(bst.x,bst.y,t,av,n);sva,ok=sa2(bst.x,bst.y,ang,ddv)
        if ok: moves.append([bst.id,sva,n]);used[bst.id]=used.get(bst.id,0)+n;cmtd.add(t.id)
    cds=[]
    for src in mine:
        a2v=av2(src)
        if a2v<10: continue
        for t in tgts:
            if t.id in cmtd: continue
            _,ddv,ta=icp2(src.x,src.y,t,av,min(a2v,50))
            n=max(int((t.ships+t.production*ta)*1.3)+1,int(t.ships*1.3)+5)
            if n>a2v or n<=t.ships+t.production*ta: continue
            r=(t.production*max(0,rem-ta)-n)/(ta+1)+t.production*2
            if t.ships<=t.production*4+3: r*=1.5
            cds.append((r,src,t,n,ddv))
    cds.sort(key=lambda x:-x[0])
    for r,src,t,n,ddv in cds:
        if t.id in cmtd or av2(src)<n: continue
        ang,dd2,_=icp2(src.x,src.y,t,av,n);sva,ok=sa2(src.x,src.y,ang,dd2)
        if not ok: continue
        moves.append([src.id,sva,n]);used[src.id]=used.get(src.id,0)+n;cmtd.add(t.id)
    return moves
print('✅ v1 ready')


## 🧪 Cell 5 — Test 1v1


In [ ]:
e1=make('orbit_wars',debug=False)
e1.run([orbital_strategist,v1_agent])
r1=[s.reward for s in e1.steps[-1]]
print(f'  {"🏆" if r1[0]==1 else "  "} KRONOS v6 : {r1[0]:+d}')
print(f'  {"🏆" if r1[1]==1 else "  "} v1        : {r1[1]:+d}')
e1.render(mode='ipython',width=800,height=600)


## 🎮 Cell 6 — 4-Player


In [ ]:
e4=make('orbit_wars',debug=False)
e4.run([orbital_strategist,v1_agent,'random',v1_agent])
r4=[s.reward for s in e4.steps[-1]]
for lb,rw in zip(['⚡ KRONOS v6','v1-A','🎲 Random','v1-B'],r4):
    print(f'  {"🏆" if rw==1 else "  "} {lb:14s} {rw:+d}')
e4.render(mode='ipython',width=800,height=600)


## 📊 Cell 7 — Tournament 20 Games


In [ ]:
import random as _rnd
N=20; wins={'v6':0,'v1':0,'rnd':0}
for g in range(N):
    agents=[orbital_strategist,v1_agent,'random',v1_agent]
    _rnd.shuffle(agents); kp=agents.index(orbital_strategist)
    et=make('orbit_wars',debug=False); et.run(agents)
    rws=[s.reward for s in et.steps[-1]]; w=rws.index(max(rws))
    if w==kp: wins['v6']+=1; wl='⚡ v6'
    elif agents[w]==v1_agent: wins['v1']+=1; wl='v1'
    else: wins['rnd']+=1; wl='🎲'
    print(f'G{g+1:02d}[v6@{kp}] {[f"{r:+d}" for r in rws]} → {wl}')
print('─'*50)
for nm,w in wins.items(): print(f'  {nm}: {w}/{N}  {"█"*(w*2)}')
wr=wins['v6']/N; elo=int(600+max(0,wr-0.25)*3800)
print(f'\n  Win rate : {wr:.0%}  |  Elo est: ~{elo}')
print(f'  {"🏆 TOP 3!" if elo>=1400 else "✅ Competitive" if elo>=1000 else "⚠️ Needs work"}')


## 💾 Cell 8 — Write main.py


In [ ]:
%%writefile main.py
"""
KRONOS v6 — "Survive First, Dominate Second"

ROOT CAUSE OF v5 LOSS (10% win rate):
  • Garrison too low → lost home planets while attacking
  • Suffocation = cosmetic (3 ships useless vs prod-5 planet)
  • No survival check → attacked when weaker than enemy nearby
  • Snipe timing imprecise → arrived same time as enemy or too late

v6 FIXES:
  1. Defensive Floor    — keep max(10%, incoming+buffer) ships always
  2. Lethal Suffocation — send production×2+1 per cycle
  3. 1-Tick Snipe       — arrive exactly T+1 after enemy lands
  4. Survival Gate      — only attack if our_ships > 2× nearest_enemy
  5. Initiative Bonus   — if step<15, hyper-aggressive (opening theory)
"""
import math

SX,SY,SR,INNER,MS = 50.0,50.0,5.0,38.0,500

class _P:
    __slots__=['id','owner','x','y','radius','ships','production']
    def __init__(self,*a):
        for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)
class _F:
    __slots__=['id','owner','x','y','angle','ships']
    def __init__(self,*a):
        for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)

def spd(n):   return min(6.0,1.0+(max(1,n)-1)*5.0/99.0)
def d2(ax,ay,bx,by): return math.sqrt((ax-bx)**2+(ay-by)**2)
def inn(p):   return d2(p.x,p.y,SX,SY)<INNER
def pred(p,av,t):
    if not inn(p): return p.x,p.y
    r=d2(p.x,p.y,SX,SY); a=math.atan2(p.y-SY,p.x-SX)+av*t
    return SX+r*math.cos(a),SY+r*math.sin(a)
def icp(sx,sy,tp,av,n,it=20):
    tx,ty=tp.x,tp.y
    for _ in range(it):
        dd=d2(sx,sy,tx,ty); t=dd/spd(n) if spd(n)>0 else 1e9
        nx,ny=pred(tp,av,t)
        if d2(tx,ty,nx,ny)<0.015: break
        tx,ty=(tx+nx)/2,(ty+ny)/2
    dd=d2(sx,sy,tx,ty); t=dd/spd(n) if spd(n)>0 else 1e9
    return math.atan2(ty-sy,tx-sx),dd,t
def sun_ok(ox,oy,a,md):
    dx,dy=math.cos(a),math.sin(a); fx,fy=SX-ox,SY-oy; tp=fx*dx+fy*dy
    if not(0<tp<md): return True
    return abs(fx*dy-fy*dx)>=SR+1.5
def safe(ox,oy,a,d,sw=42,st=36):
    if sun_ok(ox,oy,a,d): return a,True
    for i in range(1,st+1):
        da=math.radians(sw)*i/st
        for s in(+1,-1):
            alt=a+s*da
            if sun_ok(ox,oy,alt,d): return alt,True
    return a,False

# ── Binary-search minimum capture ships ──────────────────────────────────────
def capture_n(tgt,av,sx,sy,buf=1.07):
    lo,hi=1,max(tgt.ships*2+20,30)
    for _ in range(16):
        mid=(lo+hi)//2
        _,_,eta=icp(sx,sy,tgt,av,mid)
        if mid>int((tgt.ships+tgt.production*eta)*buf):
            hi=mid
        else:
            lo=mid+1
    return hi

# ── FIX 1: DEFENSIVE FLOOR ───────────────────────────────────────────────────
def min_garrison(planet, stp, incoming_ships=0):
    """
    Keep at least:
      - incoming × 1.15 + 5 (if threatened)
      - max(10% of current ships, production×2, 4) (baseline)
    This prevents losing home planets while attacking.
    """
    if incoming_ships > 0:
        return int(incoming_ships * 1.15) + 5
    floor_10pct = max(4, int(planet.ships * 0.10))
    if stp > 380:
        return max(3, floor_10pct // 2)   # endgame: minimal
    if stp < 20:
        return 4                           # opening: very aggressive
    return max(floor_10pct, planet.production * 2, 4)

# ── FIX 4: SURVIVAL GATE ─────────────────────────────────────────────────────
def survival_ok(mine, planets, player, stp):
    """
    Allow full aggression only if:
      our_total_ships >= 1.5 × nearest_enemy_total
    In opening (stp<15): always OK.
    In endgame (stp>370): always OK.
    """
    if stp < 15 or stp > 370:
        return True

    my_total = sum(p.ships for p in mine)
    my_prod  = sum(p.production for p in mine)

    cx = sum(p.x for p in mine)/max(1,len(mine))
    cy = sum(p.y for p in mine)/max(1,len(mine))

    # Find nearest enemy cluster
    enemy_ids = set(p.owner for p in planets if p.owner>=0 and p.owner!=player)
    if not enemy_ids: return True

    nearest_enemy_power = 0
    for eid in enemy_ids:
        e_planets = [p for p in planets if p.owner==eid]
        if not e_planets: continue
        e_center_dist = min(d2(p.x,p.y,cx,cy) for p in e_planets)
        if e_center_dist < 50:   # only nearby enemies matter
            ep = sum(p.ships for p in e_planets)
            nearest_enemy_power = max(nearest_enemy_power, ep)

    return my_total >= nearest_enemy_power * 1.5 or my_prod >= 8

# ── FIX 2: LETHAL SUFFOCATION ────────────────────────────────────────────────
def lethal_suffocation(enemy, mine, av, used, stp):
    """
    Send production×2+1 ships every ~12 turns to highest-production enemy.
    Goal: their garrison never builds up enough to launch.
    """
    if stp < 60: return []
    if stp % 12 not in (0, 1): return []

    out = []
    targets = sorted([p for p in enemy if p.production >= 2],
                     key=lambda p: -p.production)[:2]

    for tgt in targets:
        n = tgt.production * 2 + 1   # lethal: resets their counter fully

        src = min(
            [p for p in mine
             if p.ships - used.get(p.id,0) - min_garrison(p,stp) >= n],
            key=lambda p: d2(p.x,p.y,tgt.x,tgt.y),
            default=None
        )
        if src is None: continue

        a,dd,_ = icp(src.x,src.y,tgt,av,n)
        sa,ok  = safe(src.x,src.y,a,dd)
        if ok:
            out.append((src.id, sa, n))

    return out

# ── FIX 3: 1-TICK SNIPE ──────────────────────────────────────────────────────
def one_tick_snipe(neutral, e_fleets, mine, av, used, done, enroute, stp):
    """
    Detect enemy fleet heading to neutral.
    Send our fleet to arrive exactly T+1 turns after enemy.
    Enemy spends ships capturing it → we take it at near-zero defense.
    """
    snipes = []

    for tgt in neutral:
        if tgt.id in done or tgt.id in enroute: continue

        incoming = []
        for f in e_fleets:
            a,dd,eta = icp(f.x,f.y,tgt,av,f.ships)
            if dd < tgt.radius+4 and eta < 60:
                incoming.append((eta, f.ships))

        if not incoming: continue
        incoming.sort()
        e_eta, e_ships = incoming[0]

        # After battle
        after_def = tgt.ships + tgt.production * e_eta
        if e_ships <= after_def: continue   # enemy doesn't capture

        enemy_leftover = max(0, e_ships - after_def)

        # We arrive T+1: face only enemy_leftover + 1 turn production
        our_arrival = e_eta + 1.0
        n_need = int(enemy_leftover * 1.1) + tgt.production + 2
        n_need = max(n_need, 3)

        for src in sorted(mine, key=lambda p: d2(p.x,p.y,tgt.x,tgt.y)):
            spare = src.ships - used.get(src.id,0) - min_garrison(src,stp)
            if spare < n_need: continue

            # Use Speed-as-Timing: compute n to arrive at our_arrival
            dist = d2(src.x,src.y,tgt.x,tgt.y)
            req_spd = max(1.0, min(6.0, dist / max(1,our_arrival)))
            n_timed = max(n_need, int(1 + (req_spd-1.0)*99/5))

            if n_timed > spare: n_timed = spare
            if n_timed < n_need: continue

            a,dd,our_eta = icp(src.x,src.y,tgt,av,n_timed)
            if our_eta < e_eta + 0.5: continue   # must arrive after enemy
            if our_eta > e_eta + 4:   continue   # not too late

            sa,ok = safe(src.x,src.y,a,dd)
            if ok:
                score = tgt.production*8 + e_ships*0.4
                snipes.append((score, src.id, sa, n_timed, tgt.id))
                break

    snipes.sort(key=lambda x:-x[0])
    return snipes

# ── Score function ────────────────────────────────────────────────────────────
def score(tgt, n, eta, rem):
    tw = max(0, rem-eta)
    if tw <= 0: return -1e9
    prod = tgt.production
    s = (prod**2)*12*tw + prod*tw
    if tgt.owner >= 0: s *= 1.7        # enemy: gain + deny
    if tgt.ships <= prod*2+2: s *= 2.0 # near-empty bonus
    s -= n*0.35
    return s

# ── Main agent ────────────────────────────────────────────────────────────────
def orbital_strategist(obs):
    if isinstance(obs,dict):
        pl=obs.get('player',0); rp=obs.get('planets',[])
        rf=obs.get('fleets',[]); av=obs.get('angular_velocity',0.0366)
        stp=obs.get('step',0)
    else:
        pl=obs.player; rp=obs.planets; rf=obs.fleets
        av=obs.angular_velocity; stp=getattr(obs,'step',0)

    try:
        from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as NP,Fleet as NF
        planets=[NP(*p) for p in rp]; fleets=[NF(*f) for f in rf]
    except Exception:
        planets=[_P(*p) for p in rp]; fleets=[_F(*f) for f in rf]

    mine    = [p for p in planets if p.owner==pl]
    neutral = [p for p in planets if p.owner<0]
    enemy   = [p for p in planets if p.owner>=0 and p.owner!=pl]
    others  = enemy + neutral
    if not mine or not others: return []

    rem=MS-stp; moves=[]; used={}; done=set()
    def avail(p):  return p.ships - used.get(p.id,0)
    def rsv(pid,n): used[pid]=used.get(pid,0)+n
    def spare(p):  return avail(p) - min_garrison(p,stp)

    # ── Incoming threats ──────────────────────────────────────────────────
    incoming = {}
    for f in fleets:
        if f.owner==pl: continue
        for p in mine:
            _,dd,eta = icp(f.x,f.y,p,av,f.ships)
            if dd < p.radius+spd(f.ships)*1.5+1 and eta < 20:
                incoming[p.id] = incoming.get(p.id,0) + f.ships

    # ── DEFENSE (always runs, no survival gate) ───────────────────────────
    for p in mine:
        thr = incoming.get(p.id,0)
        if thr == 0: continue
        need = min_garrison(p,stp,thr)
        deficit = need - avail(p)
        if deficit <= 0: continue
        donors = sorted(
            [s for s in mine if s.id!=p.id and spare(s)>4],
            key=lambda s: d2(s.x,s.y,p.x,p.y)
        )
        for src in donors[:3]:
            snd = min(spare(src), deficit)
            if snd <= 0: continue
            a,dd,_ = icp(src.x,src.y,p,av,snd)
            sa,ok  = safe(src.x,src.y,a,dd)
            if ok:
                moves.append([src.id,sa,snd])
                rsv(src.id,snd); deficit-=snd
            if deficit <= 0: break

    # ── En-route ──────────────────────────────────────────────────────────
    enroute = set()
    for f in fleets:
        if f.owner!=pl: continue
        for t in others:
            _,dd,eta = icp(f.x,f.y,t,av,f.ships)
            if dd < t.radius+4 and eta < 80:
                enroute.add(t.id)

    e_fleets = [f for f in fleets if f.owner!=pl and f.owner>=0]

    # ── FIX 3: 1-TICK SNIPE ──────────────────────────────────────────────
    for sc,sid,sa,n,tid in one_tick_snipe(neutral,e_fleets,mine,av,used,done,enroute,stp)[:2]:
        src = next((p for p in mine if p.id==sid),None)
        if src and spare(src) >= n:
            moves.append([sid,sa,n]); rsv(sid,n); done.add(tid)

    # ── RETROGRADE ────────────────────────────────────────────────────────
    for p in planets:
        if p.owner<0 or p.owner==pl: continue
        if p.id in done or p.id in enroute: continue
        dep = sum(f.ships for f in fleets
                  if f.owner==p.owner and d2(f.x,f.y,p.x,p.y)<14)
        if dep < 10: continue
        if dep/max(1,p.ships+dep) < 0.30: continue
        bsrc=None; bn=0; bsa=0.0
        for src in mine:
            sp = spare(src)
            if sp < 4: continue
            n = capture_n(p,av,src.x,src.y)
            if sp < n: continue
            a,dd,_ = icp(src.x,src.y,p,av,n)
            sa,ok  = safe(src.x,src.y,a,dd)
            if not ok: continue
            if bsrc is None or sp > bn: bsrc,bn,bsa=src,n,sa
        if bsrc:
            moves.append([bsrc.id,bsa,bn]); rsv(bsrc.id,bn); done.add(p.id)

    # ── FIX 2: LETHAL SUFFOCATION ─────────────────────────────────────────
    for sid,sa,n in lethal_suffocation(enemy,mine,av,used,stp):
        src = next((p for p in mine if p.id==sid),None)
        if src and spare(src) >= n:
            moves.append([sid,sa,n]); rsv(sid,n)

    # ── FIX 4: SURVIVAL GATE → main attack ────────────────────────────────
    can_attack = survival_ok(mine,planets,pl,stp)

    # ── SCORE ALL TARGETS ─────────────────────────────────────────────────
    cands=[]
    for src in mine:
        sp = spare(src)
        if sp < 3: continue
        for tgt in others:
            if tgt.id in done or tgt.id in enroute: continue
            n = capture_n(tgt,av,src.x,src.y)
            if n > sp: continue
            a,dd,eta = icp(src.x,src.y,tgt,av,n)
            sa,ok    = safe(src.x,src.y,a,dd)
            if not ok: continue
            sc = score(tgt,n,eta,rem)
            if sc > -1e8:
                cands.append((sc,id(src),src,tgt,n,sa))

    cands.sort(key=lambda x:-x[0])

    max_atk = 6 if (stp<80 or stp>380) else 4

    atks=0
    for sc,_,src,tgt,n,sa in cands:
        if atks >= max_atk: break
        if tgt.id in done or tgt.id in enroute: continue
        if spare(src) < n: continue

        # Survival gate: neutrals always OK, enemies only if strong enough
        if tgt.owner >= 0 and not can_attack: continue

        moves.append([src.id,sa,n]); rsv(src.id,n); done.add(tgt.id); atks+=1

    # ── SWEEP: zero idle ships ────────────────────────────────────────────
    for src in sorted(mine, key=lambda p:-spare(p)):
        sp = spare(src)
        if sp < 4: continue
        best=None; bsc=-1e9
        for tgt in others:
            if tgt.id in done: continue
            n = capture_n(tgt,av,src.x,src.y)
            if n > sp: continue
            a,dd,eta = icp(src.x,src.y,tgt,av,n)
            sa,ok    = safe(src.x,src.y,a,dd)
            if not ok: continue
            if tgt.owner>=0 and not can_attack: continue
            sc2 = tgt.production*10/(dd+1) + (1.6 if tgt.owner>=0 else 1.0)
            if sc2>bsc: bsc=sc2; best=(src.id,sa,n,tgt.id)
        if best:
            moves.append([best[0],best[1],best[2]])
            rsv(best[0],best[2]); done.add(best[3])

    return moves

agent = orbital_strategist


## ✅ Cell 9 — Verify


In [ ]:
import importlib.util
spec=importlib.util.spec_from_file_location('main','main.py')
mod=importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
sub=mod.agent; print(f'✅ {sub.__name__} loaded')
ev=make('orbit_wars',debug=False)
ev.run([sub,v1_agent,'random',v1_agent])
fr=[s.reward for s in ev.steps[-1]]
print(f'Rewards: {fr}')
print('🏆 WINS!' if fr[0]==1 else '✅ Runs correctly')
